# Supplementary Figure 4 | Identification support for the N183 high-mannose glycopeptiforms and domain-resolved TMEM106B peptide QTLs

## Setup

In [ ]:
library(data.table)
library(ggplot2)
library(patchwork)

## Data: a, b

In [ ]:
# Data for a and b (figureS4ab.R, logic unchanged)
suppressMessages(library(SummarizedExperiment))
glyco <- readRDS("tmem106b/glyco/glyco_tmem106b.Rds")

genotypes <- c("C/C", "C/G", "G/G")
keep <- colData(glyco)$rs3173615 %in% genotypes & !is.na(colData(glyco)$tangles_sqrt)
glyco <- glyco[, keep]
gt <- factor(colData(glyco)$rs3173615, levels = genotypes)
stopifnot(ncol(glyco) == 316)

pairs <- list(Hex5 = c(T = "TMEM_GP04", S = "TMEM_GP02"), Hex6 = c(T = "TMEM_GP10", S = "TMEM_GP08"),
              Hex7 = c(T = "TMEM_GP16", S = "TMEM_GP14"), Hex8 = c(T = "TMEM_GP19", S = "TMEM_GP18"))
I <- assay(glyco, "intensity"); M <- assay(glyco, "match_type"); L <- assay(glyco, "localization")
localized <- matrix(grepl("N:[0-9.]+@LNN\\(1\\)", L), nrow = nrow(glyco))
direct <- I > 0 & M == "MS/MS" & assay(glyco, "spectral_count") > 0 & localized
mbr <- I > 0 & M == "MBR"
direct[is.na(direct)] <- mbr[is.na(mbr)] <- FALSE
detected <- direct | mbr
rows <- unlist(pairs)
stopifnot(!anyNA(I[rows, ]), all(I[rows, ] >= 0), all(detected[rows, ] == (I[rows, ] > 0)))

coverage <- paired <- list()
het <- gt == "C/G"
for (h in names(pairs)) {
  r <- pairs[[h]]
  coverage[[h]] <- do.call(rbind, lapply(genotypes, function(g) {
    ix <- gt == g
    do.call(rbind, lapply(names(r), function(a) {
      row <- r[a]
      data.frame(glycan = h, genotype = g, allele = paste0(a, "185"), evidence = c("MS/MS", "MBR", "Not quantified"),
                 n = c(sum(direct[row, ix]), sum(mbr[row, ix]), sum(!detected[row, ix])), N = sum(ix))
    }))
  }))
  paired[[h]] <- data.frame(glycan = h, support = c("Both quantified", "Both direct"),
                            n = c(sum(detected[r["T"], het] & detected[r["S"], het]), sum(direct[r["T"], het] & direct[r["S"], het])), N = sum(het))
}
coverage <- do.call(rbind, coverage); paired <- do.call(rbind, paired)
coverage$evidence <- factor(coverage$evidence, c("MS/MS", "MBR", "Not quantified"))
coverage$genotype <- factor(coverage$genotype, genotypes)
paired$support <- factor(paired$support, c("Both quantified", "Both direct"))
saveRDS(list(coverage = coverage, paired = paired, genotypes = genotypes, genotype_n = as.integer(table(gt)), glycans = names(pairs)), "data/Figure_S4ab.rds")
table(gt)

## Data: c

In [ ]:
# Data for c (figureS4c.R, logic unchanged)
tmt <- readRDS("tmem106b/tmt/tmt_tmem106b.Rds")

d <- as.data.frame(colData(tmt))
priority <- c("r01", "r03", "r02")
stopifnot(is.character(d$ProjID), all(nchar(d$ProjID) == 8), all(d$Round %in% priority), !anyNA(d$Batch))
ix <- order(match(d$Round, priority), d$Batch, seq_len(nrow(d)))
ix <- ix[!duplicated(d$ProjID[ix])]
d <- d[ix, ]
a <- assay(tmt, "peptides")[, ix, drop = FALSE]
d$Batch <- factor(d$Batch); d$study <- factor(d$study); d$sex <- factor(d$msex)
d$age10 <- (d$age_death - 90) / 10
d$genotype <- factor(d$rs3173615, levels = c("C/C", "C/G", "G/G"))

peptides <- c("EDAYDGVTSENMR", "GQENQLVALIPYSDQR", "SAYVSYDVQK", "YQYVDCGR")
z <- t(vapply(peptides, function(p) (a[p, ] - mean(a[p, ], na.rm = TRUE)) / sigma(lm(a[p, ] ~ Batch, data = d)), numeric(ncol(a))))
d$N2 <- colMeans(z[c("EDAYDGVTSENMR", "GQENQLVALIPYSDQR"), ], na.rm = FALSE)
d$Core <- z["SAYVSYDVQK", ]
d$Boundary <- z["YQYVDCGR", ]

plots <- tests <- list()
for (endpoint in c("N2", "Core", "Boundary")) {
  v <- d
  v$y <- v[[endpoint]]
  v <- droplevels(v[complete.cases(v[, c("y", "genotype", "Batch", "study", "age10", "sex")]), ])
  fit <- lm(y ~ genotype + Batch + study + age10 + sex, data = v)
  X <- model.matrix(fit); X <- X[, !is.na(coef(fit)), drop = FALSE]
  b <- coef(fit)[colnames(X)]; h <- hatvalues(fit); stopifnot(all(h < 1 - 1e-8))
  bread <- solve(crossprod(X))
  V <- bread %*% crossprod(X, X * (residuals(fit) / (1 - h))^2) %*% bread
  g <- grep("^genotype", colnames(X))
  W <- as.numeric(t(b[g]) %*% solve(V[g, g, drop = FALSE], b[g]))
  p <- pf(W / length(g), length(g), df.residual(fit), lower.tail = FALSE)
  tests[[endpoint]] <- data.frame(endpoint, n = nrow(v), p_genotype_HC3 = p, CG_minus_CC = b[g[1]], GG_minus_CC = b[g[2]],
                                  formula = "y ~ genotype + Batch + study + age10 + sex", row.names = NULL)
  adjusted <- residuals(fit) + as.vector(X[, g, drop = FALSE] %*% b[g])
  v$adjusted <- adjusted - mean(adjusted) + mean(v$y)
  v$endpoint <- endpoint
  plots[[endpoint]] <- v[, c("ProjID", "genotype", "adjusted", "endpoint")]
}
plot_data <- do.call(rbind, plots); results <- do.call(rbind, tests)
results$q_BH <- p.adjust(results$p_genotype_HC3, "BH")
saveRDS(list(plot_data = plot_data, results = results), "data/Figure_S4c.rds")
results[, c("endpoint", "n", "p_genotype_HC3", "CG_minus_CC", "GG_minus_CC", "q_BH")]

## Panels a and b

In [ ]:
library(ggplot2)
sab <- readRDS("data/Figure_S4ab.rds"); coverage <- sab$coverage; paired <- sab$paired; genotypes <- sab$genotypes
genotype_labels <- setNames(sprintf("%s (n=%d)", genotypes, sab$genotype_n), genotypes)
n_het <- unique(paired$N); stopifnot(length(n_het) == 1)
paired$group <- sprintf("C/G (n=%d)", n_het); paired$pair <- "pair"

# Shared colour scale: fraction of all donors in the indicated genotype group
heatmap <- function(data, x, y) {
  data$fraction <- data$n / data$N
  data$label <- sprintf("%d/%d\n%.0f%%", data$n, data$N, 100 * data$fraction)
  ggplot(data, aes(.data[[x]], .data[[y]])) +
    geom_tile(aes(fill = fraction), colour = "white", linewidth = .6) +
    geom_text(aes(label = label, colour = fraction > .62), size = 3.2, lineheight = .95) +
    scale_colour_manual(values = c("FALSE" = "#203642", "TRUE" = "white"), guide = "none") +
    scale_fill_gradient(low = "#f0f3f5", high = "#24597a", limits = c(0, 1), guide = "none") +
    labs(x = NULL, y = NULL) + theme_minimal(base_size = 11) +
    theme(panel.grid = element_blank(), panel.border = element_rect(colour = "black", fill = NA, linewidth = .4),
          strip.text.y = element_blank(), strip.text.x = element_text(face = "bold"), panel.spacing.y = unit(6, "pt"))
}
# rotated glycan composition symbols (HexNAc square, Hex circle) for one facet column (centre yc, panel height h)
sym <- function(lab, xl, yc, h) list(
  geom_text(data = lab, aes(x = xl, y = yc + .31 * h, label = "2×"), inherit.aes = FALSE, angle = -90, size = 4.6),
  geom_point(data = lab, aes(x = xl, y = yc + .105 * h), inherit.aes = FALSE, shape = 22, fill = "#2F7ED8", colour = "#1F4E79", size = 4, stroke = .6),
  geom_text(data = lab, aes(x = xl, y = yc - .105 * h, label = paste0(hex, "×")), inherit.aes = FALSE, angle = -90, size = 4.6),
  geom_point(data = lab, aes(x = xl, y = yc - .31 * h), inherit.aes = FALSE, shape = 21, fill = "#68C83E", colour = "#2E6B25", size = 4, stroke = .6))
hexn <- as.integer(sub("Hex", "", sab$glycans))
labA <- data.frame(glycan = sab$glycans, hex = hexn, genotype = genotypes[3])
p4A <- heatmap(coverage, "evidence", "allele") +
  scale_y_discrete(limits = c("S185", "T185"), expand = c(0, 0)) +
  scale_x_discrete(labels = c("MS/MS", "MBR", "Not\nquantified"), expand = c(0, 0)) +
  facet_grid(glycan ~ genotype, labeller = labeller(genotype = genotype_labels)) +
  sym(labA, 3.75, 1.5, 2) + coord_cartesian(xlim = c(0.5, 3.5), ylim = c(0.5, 2.5), clip = "off") + theme(plot.margin = margin(5.5, 38, 5.5, 5.5))
labB <- data.frame(glycan = sab$glycans, hex = hexn, group = paired$group[1])
p4B <- heatmap(paired, "support", "pair") +
  scale_x_discrete(labels = c("Both\nquantified", "Both\nMS/MS"), expand = c(0, 0)) + scale_y_discrete(expand = c(0, 0)) +
  facet_grid(glycan ~ group) + sym(labB, 2.68, 1, 1) +
  coord_cartesian(xlim = c(0.5, 2.5), clip = "off") + theme(axis.text.y = element_blank(), plot.margin = margin(5.5, 44, 5.5, 0))
options(repr.plot.width = 9, repr.plot.height = 6); p4A | p4B

## Panel c

In [ ]:
sc <- readRDS("data/Figure_S4c.rds"); plot_data <- sc$plot_data; results <- sc$results
fmtP <- function(p) { if (p < 2.2e-16) return('"Genotype P < 2 × 10"^-16'); e <- floor(log10(p)); if (e >= -2) return(sprintf('"Genotype P = %s"', signif(p, 2)))
  sprintf('"Genotype P = %s × 10"^%d', signif(p / 10^e, 2), e) }
titles <- c(N2 = 'N[mean]~"score"', Core = 'C[core]~"peptide: SAYVSYDVQK"', Boundary = 'C[boundary]~"peptide: YQYVDCGR"')
labels <- setNames(sprintf('atop(bold(%s), bold(%s))', titles[results$endpoint], sapply(results$p_genotype_HC3, fmtP)), results$endpoint)
plot_data$endpoint <- factor(plot_data$endpoint, levels = names(titles))
# Figure 5 style: genotype order C/C, C/G, G/G
plot_data$genotype <- factor(as.character(plot_data$genotype), levels = c("C/C", "C/G", "G/G"))
geno_cols <- c("G/G" = "#1f78b4", "C/G" = "#ff7f00", "C/C" = "#33a02c")
counts <- aggregate(adjusted ~ endpoint + genotype, plot_data, length)
med <- aggregate(adjusted ~ endpoint + genotype, plot_data, median); med$x <- as.numeric(med$genotype)

p4C <- ggplot(plot_data, aes(genotype, adjusted)) +
  geom_boxplot(width = 0.5, alpha = 0.7, outlier.shape = NA, colour = "black") +
  geom_jitter(aes(colour = genotype), width = 0.2, height = 0, size = 1.6, alpha = 0.6) +
  geom_line(data = med, aes(x = x, y = adjusted, group = endpoint), inherit.aes = FALSE, colour = "grey50", linewidth = 0.8, linetype = "dashed") +
  geom_text(data = counts, aes(y = Inf, label = paste0("n=", adjusted)), vjust = 1.3, colour = "black", size = 3.4) +
  facet_wrap(~ endpoint, scales = "free_y", labeller = labeller(endpoint = as_labeller(labels, label_parsed)), nrow = 1) +
  scale_colour_manual(values = geno_cols) +
  scale_y_continuous(expand = expansion(mult = c(.05, .15))) +
  labs(x = "rs3173615 genotype (C = T185; G = S185)", y = "Normalized abundance") +
  theme_bw(base_size = 12) +
  theme(legend.position = "none", panel.grid = element_blank(), panel.border = element_rect(linewidth = 1.0),
        strip.background = element_blank(), strip.text = element_text(face = "bold"),
        axis.title = element_text(face = "bold"), axis.text = element_text(colour = "black"))
options(repr.plot.width = 10, repr.plot.height = 4.5); p4C

## Combine & save

In [ ]:
library(patchwork)
# a | b on top (b carries the shared rotated glycan labels), c below
combined_fig <- ((p4A | p4B) + plot_layout(widths = c(6.8, 2.05))) / p4C + plot_layout(heights = c(6.1, 3.0)) +
  plot_annotation(tag_levels = 'a') & theme(plot.tag = element_text(size = 18, face = 'bold'), plot.tag.position = c(0.01, 0.99))
options(repr.plot.width = 9, repr.plot.height = 9.1); combined_fig

ggsave('Supplementary_Figure4.pdf', combined_fig, device = grDevices::cairo_pdf, width = 9, height = 9.1, bg = 'white')
cat('Saved: Supplementary_Figure4.pdf\n')